## Bronze Table: Lines

Ingests data from the `lines` volume and performs an **upsert (merge)** into the `bronze.lines` Delta table based on the line `id`. 

- **New records** are inserted with their current timestamp as `last_update_date`
- **Existing records** are updated only when changes are detected, and `last_update_date` is refreshed to the current timestamp

Columns containing only null values are dropped to prevent schema conflicts. Empty arrays are converted to empty strings for schema compatibility.

**Key columns:**
- `id` - Unique identifier used for merge logic
- `last_update_date` - Timestamp of the last insert or update for each record

In [0]:
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import current_timestamp, to_date, lit
from delta.tables import DeltaTable

In [0]:
# 1. Read and clean data
file_path = "/Volumes/trips_carris_metropolitana/bronze/lines/lines.json"
df = pd.read_json(file_path)
df = df.dropna(axis=1, how='all')
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, list) and len(x) == 0).all():
        df[col] = ""

# 2. Convert to Spark DataFrame
spark_df = spark.createDataFrame(df)

# 3. Add last_update_date
spark_df = spark_df.withColumn("last_update_date", current_timestamp())

table_name = "trips_carris_metropolitana.bronze.lines"

# 4. Check if table exists
if not spark.catalog.tableExists(table_name):
    # First run: create the Delta table
    spark_df.write.format("delta").saveAsTable(table_name)
    print(f"Created table {table_name} with {df.shape[0]} records")
else:
    # Perform merge (upsert) based on 'id'
    delta_table = DeltaTable.forName(spark, table_name)
    delta_table.alias("target").merge(
        source=spark_df.alias("source"),
        condition="target.id = source.id"
    ).whenMatchedUpdate(set={
        # Update all columns
        **{col: f"source.{col}" for col in spark_df.columns if col != "last_update_date"},
        "last_update_date": "source.last_update_date"
    }).whenNotMatchedInsert(values={
        **{col: f"source.{col}" for col in spark_df.columns if col != "last_update_date"},
        "last_update_date": "source.last_update_date"
    }).execute()
    print(f"Upserted data into {table_name}")

In [0]:
%sql
SELECT COUNT(*) FROM trips_carris_metropolitana.bronze.lines;

In [0]:
%sql
SELECT * FROM trips_carris_metropolitana.bronze.lines LIMIT 5;